In [1]:
import os

# Change working root directory
try:
    os.chdir("/detectives_minimal_implementation")
except:
    pass

from booknlp_fr import (
    load_tokenizer_and_embedding_model,
    get_embedding_tensor_from_tokens_df,
    load_text_file,
    load_tokens_df,
    save_tokens_df,
    load_entities_df,
)
from tqdm.auto import tqdm
import torch
import os
import pandas as pd
from collections import Counter
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, classification_report, confusion_matrix

/home/antoine/Bureau/character_attributes_classification/.venv/lib/python3.10/site-packages/booknlp_fr/booknlp_fr_add_entities_features.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


booknlp_fr package loaded successfully.


In [2]:
detectives_features_dataset_path = os.path.join("data", "detectives_features_dataset.pt")

# ✅ Load it back
detectives_features_dataset = torch.load(detectives_features_dataset_path)

## Generate average embedding from all attributes

In [3]:
att_types = ['agent_embeddings', 'mod_embeddings', 'pos_embeddings'] # 'patient_embeddings' is non informative
for i, char_features_dict in enumerate(tqdm(detectives_features_dataset)):
    # Concatenate all relevant embeddings
    all_attributes_embeddings = torch.concat([char_features_dict[att_type] for att_type in att_types], dim=0)

    mean_embedding = torch.mean(all_attributes_embeddings, dim=0)
    mean_embedding = torch.nan_to_num(mean_embedding, nan=0.0)
    detectives_features_dataset[i]["mean_embedding"] = mean_embedding

torch.save(detectives_features_dataset, detectives_features_dataset_path)

  0%|          | 0/604 [00:00<?, ?it/s]

In [4]:
multi_doc_characters = [
    {"character_name": "le_comissaire_maigret", "authors": ["Simenon-Georges"], "aliases": ["maigret", "le commissaire maigret", "monsieur maigret"]},
    {"character_name": "juve", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["juve"]},
    {"character_name": "rouletabille", "authors": ["Leroux-Gaston"], "aliases": ["rouletabille"]},
    {"character_name": "fantômas", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["fantômas"]},
    {"character_name": "jérôme_fandor", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["fandor", "jérôme fandor"]},
    {"character_name": "arsène_lupin", "authors": ["Leblanc-Maurice"], "aliases": ["arsène lupin", "lupin", "raoul", "jim barnett", "barnett", "don luis", "victor"]},
    {"character_name": "adamsberg", "authors": ["Vargas-Fred"], "aliases": ["adamsberg"]},
    {"character_name": "monsieur_lecoq", "authors": ["Gaboriau-Emile"], "aliases": ["lecoq", "m. lecoq", "m. verduret"]},
    {"character_name": "inspecteur_ganimard", "authors": ["Leblanc-Maurice"], "aliases": ["ganimard"]},
    {"character_name": "inspecteur_cadin", "authors": ["Daeninckx-Didier"], "aliases": ["cadin"]},
    {"character_name": "gabriel_lecouvreur", "authors": ["Daeninckx-Didier"], "aliases": ["gabriel"]},
    {"character_name": "camille", "authors": ["Lemaitre-Pierre"], "aliases": ["camille"]},
    {"character_name": "inspecteur_ménardier", "authors": ["Bernede-Arthur"], "aliases": ["ménardier"]},
    {"character_name": "inspecteur_huret", "authors": ["Bouvier-Alexis"], "aliases": ["l' agent huret"]},
    {"character_name": "chéri_bibi", "authors": ["Leroux-Gaston"], "aliases": ["chéri - bibi"]},
    {"character_name": "juge_denizet", "authors": ["Zola-Emile"], "aliases": ["m. denizet"]},
    {"character_name": "herlock_sholmes", "authors": ["Leblanc-Maurice"], "aliases": ["sholmès"]},
    {"character_name": "isidore_beautrelet", "authors": ["Leblanc-Maurice"], "aliases": ["isidore"]},
    {"character_name": "prasville_secrétaire_général_préfecture", "authors": ["Leblanc-Maurice"], "aliases": ["prasville"]},
    {"character_name": "père_tabaret", "authors": ["Gaboriau-Emile"], "aliases": ["le père tabaret"]},
    {"character_name": "père_plantat", "authors": ["Gaboriau-Emile"], "aliases": ["le père plantat"]},
    {"character_name": "nestor_burma", "authors": ["Malet-Leo"], "aliases": ["monsieur burma", "nestor burma"]},
    {"character_name": "john_jarvis", "authors": ["Le-Rouge-Gustave"], "aliases": ["john_jarvis"]},
    {"character_name": "wallas", "authors": ["Robbe-Grillet-Alain"], "aliases": ["wallas"]},
    {"character_name": "san_antonio", "authors": ["San-Antonio"], "aliases": ["san-antonio"]},
    {"character_name": "jobin", "authors": ["Montepin-Xavier-de"], "aliases": ["jobin"]},
    {"character_name": "commissaire_niémans", "authors": ["Grange-Jean-Christophe"], "aliases": ["niémans"]},
    {"character_name": "john_jarvis", "authors": ["Le-Rouge-Gustave"], "aliases": ["john jarvis"]},
    {"character_name": "théodore_béchoux", "authors": ["Leblanc-Maurice"], "aliases": ["béchoux"]},
    {"character_name": "monsieur_desmalions", "authors": ["Leblanc-Maurice"], "aliases": ["m. desmalions"]},
    {"character_name": "monsieur_hilaire", "authors": ["Leroux-Gaston"], "aliases": ["m. hilaire"]},
    {"character_name": "serge_rénine", "authors": ["Leblanc-Maurice"], "aliases": ["rénine"]},
    {"character_name": "hercule_petitgris", "authors": ["Leblanc-Maurice"], "aliases": ["petitgris"]},
    {"character_name": "patricia_johnston", "authors": ["Leblanc-Maurice"], "aliases": ["patricia"]},
    {"character_name": "monsieur_bessières", "authors": ["Leroux-Gaston"], "aliases": ["m. bessières"]},
    {"character_name": "monsieur_lebouc", "authors": ["Leroux-Gaston"], "aliases": ["lebouc"]},
    {"character_name": "cravely", "authors": ["Leroux-Gaston"], "aliases": ["cravely"]},
    {"character_name": "commissaire_delvigne", "authors": ["Simenon-Georges"], "aliases": ["le commissaire delvigne"]},
    {"character_name": "inspecteur_boutigues", "authors": ["Simenon-Georges"], "aliases": ["boutigues"]},
    {"character_name": "torrence", "authors": ["Simenon-Georges"], "aliases": ["torrence"]},
    {"character_name": "richard_lafargue", "authors": ["Jonquet-Thierry"], "aliases": ["richard"]},
    {"character_name": "rené_griffon", "authors": ["Daeninckx-Didier"], "aliases": ["griffon"]},
    {"character_name": "alain_deligny", "authors": ["Daeninckx-Didier"], "aliases": ["alain deligny"]},
    {"character_name": "michèle_fogel", "authors": ["Daeninckx-Didier"], "aliases": ["michèle fogel"]},
    {"character_name": "yves guyot", "authors": ["Daeninckx-Didier"], "aliases": ["guyot", "yves guyot"]},
    {"character_name": "commissaire_divisionnaire_darqué", "authors": ["Daeninckx-Didier"], "aliases": ["darqué"]},
    {"character_name": "inspecteur_divisionnaire_darqué", "authors": ["Jonquet-Thierry"], "aliases": ["rovère"]},
    {"character_name": "colonel_starr", "authors": ["Gary-Romain"], "aliases": ["starr"]},
    {"character_name": "monsignor_domani", "authors": ["Gary-Romain"], "aliases": ["monsignor domani"]},
    {"character_name": "adrien_danglard", "authors": ["Vargas-Fred"], "aliases": ["danglard"]},
    {"character_name": "bouzille", "authors": ["Allain-Marcel-&-Souvestre-Pierre"], "aliases": ["bouzille"]},
]

char_name_to_multidoc_mapping = {}

for multidoc_character in multi_doc_characters:
    multidoc_alias = multidoc_character["character_name"]
    for author in multidoc_character["authors"]:
        for alias in multidoc_character["aliases"]:
            char_name_to_multidoc_mapping[f"{author} {alias}"] = multidoc_alias

print(len(char_name_to_multidoc_mapping))

64


In [5]:
for char_id, char_dict in enumerate(detectives_features_dataset):
    file_name = char_dict["file_name"]
    year = int(file_name.split("_")[0])
    author = file_name.split("_")[1]
    char_name = char_dict["char_name"]
    author_alias = f"{author} {char_name}"
    if author_alias in char_name_to_multidoc_mapping:
        multidoc_alias = char_name_to_multidoc_mapping[author_alias]
    else:
        multidoc_alias = author_alias
    char_dict["year"] = year
    char_dict["author"] = author
    char_dict["multidoc_alias"] = multidoc_alias

    char_dict["10year"] = year // 10
    char_dict["20year"] = year // 20
    char_dict["50year"] = year // 50

    detectives_features_dataset[char_id] = char_dict

In [6]:
X = np.array([char_features_dict["mean_embedding"] for char_features_dict in detectives_features_dataset])
y = np.array([char_features_dict["label"] for char_features_dict in detectives_features_dataset])

ids = list(range(len(char_features_dict)))

print(f"Detective Counter: {Counter(y)}")

Detective Counter: Counter({0: 419, 1: 185})


In [7]:
def get_evaluation_results_as_dict(gol_labels, predicted_labels, model_name):
    report_dict = classification_report(gol_labels, predicted_labels, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()

    results_dict = {
        "model_name": model_name,
        "balanced_accuracy": report_df.loc["macro avg", "recall"],
         "non_detective_precision": report_df.loc["0", "precision"],
         "non_detective_recall": report_df.loc["0", "recall"],
         "non_detective_f1_score": report_df.loc["0", "f1-score"],
         "detective_precision": report_df.loc["1", "precision"],
         "detective_recall": report_df.loc["1", "recall"],
         "detective_f1_score": report_df.loc["1", "f1-score"]
                    }
    return results_dict

In [8]:
logo_results = []

In [9]:
# Set up k-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# Define the model
logistic_model = LogisticRegression(max_iter=1000,
                                    random_state=0,
                                    C=5,
                                    penalty='l2',
                                    class_weight='balanced',
                                    )
svc_model = SVC(
    random_state=42,
    probability=True,
    C = 10,
    class_weight = "balanced",
    degree = 2,
    gamma = 0.1,
    kernel = 'rbf',
)

for model_name, model in [["logistic_regression", logistic_model],
                          ["svc", svc_model]]:

    # model = svc_model
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
    # Get predicted labels
    y_pred = cross_val_predict(model, X, y, cv=cv, method='predict')

    results_dict = get_evaluation_results_as_dict(y, y_pred, model_name)

    logo_results.append(results_dict)


pd.DataFrame(logo_results)

,model_name,balanced_accuracy,non_detective_precision,non_detective_recall,non_detective_f1_score,detective_precision,detective_recall,detective_f1_score
0,logistic_regression,0.897091,0.958442,0.880668,0.917910,0.771689,0.913514,0.836634
1,svc,0.911411,0.959698,0.909308,0.933824,0.816425,0.913514,0.862245


## LOGO Cross-Validation

In [17]:
svc_model = SVC(C=1, class_weight='balanced', degree=2, gamma=1, probability=True, random_state=42)

model = svc_model

In [18]:
for logo_grouping in ['author', 'multidoc_alias', '10year', '20year', '50year']:

    groups = np.array([char_dict[logo_grouping] for char_dict in detectives_features_dataset])

    logo = LeaveOneGroupOut()
    y_true_all = []
    y_pred_all = []

    for train_idx, val_idx in tqdm(logo.split(X, y, groups), total=len(np.unique(groups))):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        # extend gold and predicted labels
        y_true_all.extend(y_val)
        y_pred_all.extend(y_pred)

    results_dict = get_evaluation_results_as_dict(y_true_all, y_pred_all, logo_grouping)
    logo_results.append(results_dict)

pd.DataFrame(logo_results)

  0%|          | 0/155 [00:00<?, ?it/s]

  0%|          | 0/447 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

,model_name,balanced_accuracy,non_detective_precision,non_detective_recall,non_detective_f1_score,detective_precision,detective_recall,detective_f1_score
0,logistic_regression,0.897091,0.958442,0.880668,0.917910,0.771689,0.913514,0.836634
1,svc,0.911411,0.959698,0.909308,0.933824,0.816425,0.913514,0.862245
2,author,0.910779,0.955335,0.918854,0.936740,0.830846,0.902703,0.865285
3,multidoc_alias,0.900600,0.950125,0.909308,0.929268,0.812808,0.891892,0.850515
4,10year,0.905373,0.950617,0.918854,0.934466,0.829146,0.891892,0.859375
5,20year,0.909585,0.955224,0.916468,0.935445,0.826733,0.902703,0.863049
6,50year,0.908882,0.942993,0.947494,0.945238,0.879781,0.870270,0.875000
7,author,0.907689,0.942857,0.945107,0.943981,0.875000,0.870270,0.872629
8,multidoc_alias,0.901406,0.940191,0.937947,0.939068,0.860215,0.864865,0.862534
9,10year,0.914604,0.949519,0.942721,0.946108,0.872340,0.886486,0.879357


## Select the best model to be trained on the full annotated dataset

In [19]:
final_model = svc_model

print(f"The model is trained with the average of: {att_types}")
print(f"The final model is trained on {len(X)} examples ({np.sum(y == 1)} detectives & {np.sum(y == 0)} non-detectives)")
print(f"Final model architecture: {final_model}")

# Training the final model
final_model.fit(X, y)

final_model_dict = {"att_types": att_types,
                    "model": final_model,}

The model is trained with the average of: ['agent_embeddings', 'mod_embeddings', 'pos_embeddings']
The final model is trained on 604 examples (185 detectives & 419 non-detectives)
Final model architecture: SVC(C=1, class_weight='balanced', degree=2, gamma=1, probability=True,
    random_state=42)


In [20]:
import joblib

detective_detector_model_path = "detective_detector_model.pkl"
# Save dict
joblib.dump(final_model_dict, detective_detector_model_path)
print(f"The DetectiveDetector model has been successfully saved to {detective_detector_model_path}")

The DetectiveDetector model has been successfully saved to detective_detector_model.pkl


In [21]:
import joblib

detective_detector_dict = joblib.load(detective_detector_model_path)
detective_detector_model = detective_detector_dict["model"]
detective_detector_att_types = detective_detector_dict["att_types"]

print(detective_detector_model)

SVC(C=1, class_weight='balanced', degree=2, gamma=1, probability=True,
    random_state=42)


In [22]:
predictions = detective_detector_model.predict(X)

In [23]:
print(Counter(predictions))

Counter({0: 403, 1: 201})
